# AKANE E9-1（続き）：確かめの答えだけを小さなファイルにする
E9-1_results.json が大きく（15MB）こちらで受け取れないため、辞書の確かめの答えだけを取り出します。GPUは要りません（1〜2分）。
「すべてのセルを実行」→ ドライブの `AKANE_アーキテクチャ/E9-1_results/E91_checker_small.json` ができます。

In [ ]:
import json, os
import numpy as np
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/AKANE_アーキテクチャ/E9-1_results"
DEPTH = 3
R = json.load(open(f"{OUT}/E9-1_results.json"))

In [ ]:
# 中身（sims/e9/e91_checker.py と同じ）
L = R['L']; CATS = R['meta']['cats']; cidx = {c: j for j, c in enumerate(CATS)}
nid = {tuple(x): i for i, x in enumerate(R['nodes'])}
AE = np.array(R['arrive_e'], dtype=np.float32)
ST = {(w, k): e for w, k, e, c in R['steps']}
out = {'e8D': {}, 'e8one': {}}
score = {'e8D': {}, 'e8one': {}}
for w, (nd, lk) in R['chains'].items():
    nodes = [tuple(x) for x in nd]
    if not nodes:
        continue
    ch = {x: [] for x in nodes}
    for a, b in lk:
        ch[tuple(a)].append(tuple(b))
    roots = [x for x in nodes if x[0] == w]
    for C in CATS:
        j = cidx[C]
        arr = {x: (1.0 if x[0] == C else float(AE[nid[x], j])) for x in nodes}
        memo = {}
        def f(x, d):
            if (x, d) in memo:
                return memo[(x, d)]
            best = arr[x]
            if best < 1.0 and d > 0:
                e = ST.get(x, 0.0)
                for c in ch[x]:
                    best = max(best, e * f(c, d - 1))
            memo[(x, d)] = best
            return best
        s6 = max(f(r, DEPTH) for r in roots) if roots else 0.0
        s1 = max(arr[r] for r in roots) if roots else 0.0
        key = f"{w}|{C}"
        score['e8D'][key] = round(s6, 4); score['e8one'][key] = round(s1, 4)
        out['e8D'][key] = int(s6 > 0.5); out['e8one'][key] = int(s1 > 0.5)

In [ ]:
ENTS = sorted(R['chains']); CATS = R['meta']['cats']
small = {'meta': {k: v for k, v in R['meta'].items()}, 'ents': ENTS, 'cats': CATS, 'checkers': {}, 'scores_e8D': {}, 'scores_e8one': {}}
for name in out:
    small['checkers'][name] = ''.join(''.join(str(out[name][f"{w}|{C}"]) if f"{w}|{C}" in out[name] else '-' for C in CATS) for w in ENTS)
for name in score:
    small['scores_' + name] = [int(round(score[name].get(f"{w}|{C}", -0.001) * 1000)) for w in ENTS for C in CATS]
small['n_nodes'] = len(R['nodes']); small['n_steps'] = len(R['steps'])
small['choice_first_rate'] = float(np.mean([v == 0 for w, k, v in R.get('choice_D', [[0, 0, 0]])]))
p = f"{OUT}/E91_checker_small.json"
json.dump(small, open(p, "w"))
print("保存:", p, round(os.path.getsize(p) / 1e6, 2), "MB")
print({k: (v.count('1'), v.count('0'), v.count('-')) for k, v in small['checkers'].items()})